# **Spoof Target VAD Filtering**

This notebook applies **Silero VAD-based validation** to processed `spoof_targets` recordings before spoof generation. The pipeline is executed separately for the following splits:

```python
splits = [
    "cv_train", "cv_val", "cv_test",
    "myst_train", "myst_val", "myst_test",
    "vox_train", "vox_val", "vox_test"
]
```

The currently processed split is controlled using:

```python
split = "vox_val"
```

To process another split, only the `split` variable needs to be changed.

The pipeline loads the processed spoof target manifest, extracts the corresponding local audio files, applies parallel Silero VAD processing, computes total detected speech duration, saves split-level VAD results, and merges the VAD metadata back into the manifest.

### **Main Configuration**

```python
TARGET_SR = 16000
VAD_THRESHOLD = 0.5
MIN_SPEECH_MS = 250
SPEECH_THRESHOLD = 10.0
NUM_WORKERS = 4
```

This version is optimized for faster execution using parallel processing with `NUM_WORKERS = 4`, which may require higher RAM, stronger CPU resources, or Colab Pro runtimes for stable processing on large splits. If runtime instability occurs, reduce the `NUM_WORKERS` value in the configuration section.

### **Generated Outputs**

The pipeline generates:
- split-level VAD result CSV files
- merged manifests containing VAD metadata for each split
- a final combined spoof target manifest:

```text
file_manifest_spoof_targets_ge3_with_vad.csv
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# copy tar from Drive to local runtime
!cp \
"/content/drive/MyDrive/age verification/processed/data/spoof_targets.tar" \
/content/

In [ ]:
split = "vox_val"

!mkdir -p /content/spoof_targets_extracted

!tar -xf \
"/content/spoof_targets.tar" \
-C "/content/spoof_targets_extracted" \
spoof_targets/{split}

In [ ]:
# ============================================================
# Parallel Silero VAD Pipeline
# Processed Audio + Local Paths
# ============================================================

!pip install -q tqdm pandas

import os
import time
import torch
import torchaudio
import pandas as pd
import warnings
import logging

from concurrent.futures import ProcessPoolExecutor, as_completed

# ============================================================
# CONFIG
# ============================================================

TARGET_SR = 16000

VAD_THRESHOLD = 0.5
MIN_SPEECH_MS = 250

SPEECH_THRESHOLD = 10.0

CHECKPOINT_EVERY = 100
NUM_WORKERS = 4

# ============================================================
# PATHS
# ============================================================

src_csv_root = "/content/drive/MyDrive/age verification/processed/manifest/spoof_targets/file_manifest_"

tgt_csv_root = "/content/drive/MyDrive/age verification/processed/manifest/spoof_targets/ge3/file_manifest_"

MANIFEST_CSV = tgt_csv_root + split + "_ge3.csv"

OUTPUT_CSV = tgt_csv_root + split + "_ge3_vad_results.csv"

# ============================================================
# Setup
# ============================================================

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

# ============================================================
# Cache Silero VAD
# ============================================================

SAFE_HUB_DIR = "/tmp/torch_hub_silero"

os.makedirs(SAFE_HUB_DIR, exist_ok=True)

torch.hub.set_dir(SAFE_HUB_DIR)

print("Pre-caching Silero VAD...")

temp_model, _ = torch.hub.load(
    "snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True,
    force_reload=False,
    verbose=False
)

del temp_model

print("Cache ready.")

# ============================================================
# Worker Globals
# ============================================================

_vad_model = None
_vad_utils = None

# ============================================================
# Worker Init
# ============================================================

def safe_init_worker():

    global _vad_model
    global _vad_utils

    for attempt in range(2):

        try:

            force = (attempt == 1)

            _vad_model, _vad_utils = torch.hub.load(
                "snakers4/silero-vad",
                model="silero_vad",
                trust_repo=True,
                force_reload=force,
                verbose=False
            )

            _vad_model.to("cpu")
            _vad_model.eval()

            return

        except Exception as e:

            if attempt == 0:

                print(
                    f"Worker cache glitch, retrying... ({e})"
                )

            else:

                raise RuntimeError(
                    f"Failed to load Silero VAD: {e}"
                )

# ============================================================
# Process One File
# ============================================================

def process_file(args):

    file_id, local_path, original_source_path = args

    try:

        if not os.path.exists(local_path):

            return {
                'file_id': file_id,
                'source_path': original_source_path,
                'raw_duration_sec': 0.0,
                'speech_duration_sec': 0.0,
                'flagged': 'MISSING',
                'vad_status': 'File not found',
                'max_amplitude': 0.0
            }

        # ----------------------------------------------------
        # Load Processed Audio
        # ----------------------------------------------------

        wav, sr = torchaudio.load(local_path)

        # Already processed:
        # - mono
        # - 16kHz

        if sr != TARGET_SR:
            raise ValueError(f"Unexpected sample rate: {sr}")

        if wav.shape[0] != 1:
            raise ValueError(
                f"Expected mono audio, got {wav.shape[0]} channels"
            )

        wav = wav.squeeze(0).to(torch.float32)

        # ----------------------------------------------------
        # Duration
        # ----------------------------------------------------

        raw_dur = len(wav) / TARGET_SR

        # ----------------------------------------------------
        # Peak Amplitude
        # ----------------------------------------------------

        amp_max = wav.abs().max().item()

        # Normalize clipped audio
        if amp_max > 1.0:

            wav = wav / amp_max
            amp_max = 1.0

        # Skip near-empty audio
        if amp_max < 0.01:

            return {
                'file_id': file_id,
                'source_path': original_source_path,
                'raw_duration_sec': round(raw_dur, 2),
                'speech_duration_sec': 0.0,
                'flagged': 'EMPTY_AUDIO',
                'vad_status': f'Low amp: {amp_max:.4f}',
                'max_amplitude': amp_max
            }

        # ----------------------------------------------------
        # Run Silero VAD
        # ----------------------------------------------------

        with torch.no_grad():

            ts = _vad_utils[0](
                wav,
                _vad_model,
                sampling_rate=TARGET_SR,
                threshold=VAD_THRESHOLD,
                min_speech_duration_ms=MIN_SPEECH_MS
            )

        # ----------------------------------------------------
        # Total Speech Duration
        # ----------------------------------------------------

        speech_dur = sum(
            (t['end'] - t['start']) / TARGET_SR
            for t in ts
        )

        return {

            'file_id': file_id,

            'source_path': original_source_path,

            'raw_duration_sec': round(raw_dur, 2),

            'speech_duration_sec': round(speech_dur, 2),

            'flagged': speech_dur < SPEECH_THRESHOLD,

            'vad_status': 'success',

            'max_amplitude': amp_max
        }

    except Exception as e:

        return {
            'file_id': file_id,
            'source_path': original_source_path,
            'raw_duration_sec': 0.0,
            'speech_duration_sec': 0.0,
            'flagged': 'ERROR',
            'vad_status': str(e),
            'max_amplitude': 0.0
        }

# ============================================================
# Main Pipeline
# ============================================================

def run_parallel_pipeline():

    # --------------------------------------------------------
    # Load Manifest
    # --------------------------------------------------------

    df = pd.read_csv(MANIFEST_CSV)

    valid_mask = (
        (df['is_corrupted'] != True)
        &
        (df['processed_path'].notna())
    )

    df_to_proc = df[valid_mask].copy()

    # --------------------------------------------------------
    # Convert processed_path -> local path
    # --------------------------------------------------------

    df_to_proc["local_path"] = (
        df_to_proc["processed_path"]
        .str.replace(
            "/content/drive/MyDrive/age verification/processed/data/spoof_targets",
            "/content/spoof_targets_extracted/spoof_targets",
            regex=False
        )
    )

    print(
        df_to_proc[
            ["processed_path", "local_path"]
        ].head()
    )

    print(
        "First file exists:",
        os.path.exists(
            df_to_proc["local_path"].iloc[0]
        )
    )

    # --------------------------------------------------------
    # Resume Logic
    # --------------------------------------------------------

    if os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 0:

        processed_ids = set(
            pd.read_csv(
                OUTPUT_CSV,
                usecols=['file_id']
            )['file_id'].tolist()
        )

        df_to_proc = df_to_proc[
            ~df_to_proc['file_id'].isin(processed_ids)
        ]

        print(
            f"Resuming: {len(processed_ids)} done. "
            f"{len(df_to_proc)} remaining."
        )

    else:

        cols = [
            'file_id',
            'source_path',
            'raw_duration_sec',
            'speech_duration_sec',
            'flagged',
            'vad_status',
            'max_amplitude'
        ]

        pd.DataFrame(columns=cols).to_csv(
            OUTPUT_CSV,
            index=False
        )

        print("Output initialized.")

    # --------------------------------------------------------
    # Prepare Tasks
    # --------------------------------------------------------

    tasks = []

    for _, row in df_to_proc.iterrows():

        tasks.append(
            (
                row["file_id"],
                row["local_path"],    # للقراءة الفعلية
                row["source_path"]
            )
        )

    print(
        f"Starting Parallel Processing "
        f"({NUM_WORKERS} workers)..."
    )

    start_time = time.time()

    batch_results = []

    # --------------------------------------------------------
    # Parallel Execution
    # --------------------------------------------------------

    try:

        with ProcessPoolExecutor(
            max_workers=NUM_WORKERS,
            initializer=safe_init_worker
        ) as executor:

            futures = {
                executor.submit(process_file, task): task
                for task in tasks
            }

            for i, future in enumerate(
                as_completed(futures),
                1
            ):

                batch_results.append(future.result())

                # ------------------------------------------------
                # Checkpoint Save
                # ------------------------------------------------

                if (
                    i % CHECKPOINT_EVERY == 0
                    or
                    i == len(futures)
                ):

                    pd.DataFrame(batch_results).to_csv(
                        OUTPUT_CSV,
                        mode='a',
                        header=False,
                        index=False
                    )

                    elapsed = (
                        time.time() - start_time
                    ) / 60

                    rate = i / elapsed if elapsed > 0 else 0

                    eta = (
                        (len(tasks) - i) / rate
                        if rate > 0 else 0
                    )

                    print(
                        f"Checkpoint "
                        f"{i}/{len(tasks)} | "
                        f"{rate:.1f} files/min | "
                        f"ETA: {eta:.0f} min"
                    )

                    batch_results.clear()

    finally:

        if batch_results:

            pd.DataFrame(batch_results).to_csv(
                OUTPUT_CSV,
                mode='a',
                header=False,
                index=False
            )

            print(
                f"Final flush "
                f"({len(batch_results)} items)"
            )

    print("Processing Complete.")

# ============================================================
# Run
# ============================================================

run_parallel_pipeline()

# ============================================================
# Summary
# ============================================================

results = pd.read_csv(OUTPUT_CSV)

print(f"\nSummary:")
print(f"Total Files: {len(results)}")

print(
    f"Flagged (<10s speech): "
    f"{results['flagged'].isin([True, 'True']).sum()}"
)

print(
    f"Saved columns are "
    f"{results.columns.tolist()} "
    f"to: {OUTPUT_CSV}"
)

Pre-caching Silero VAD...
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /tmp/torch_hub_silero/master.zip
Cache ready.
                                      processed_path  \
0  /content/drive/MyDrive/age verification/proces...   
1  /content/drive/MyDrive/age verification/proces...   
2  /content/drive/MyDrive/age verification/proces...   
3  /content/drive/MyDrive/age verification/proces...   
4  /content/drive/MyDrive/age verification/proces...   

                                          local_path  
0  /content/spoof_targets_extracted/spoof_targets...  
1  /content/spoof_targets_extracted/spoof_targets...  
2  /content/spoof_targets_extracted/spoof_targets...  
3  /content/spoof_targets_extracted/spoof_targets...  
4  /content/spoof_targets_extracted/spoof_targets...  
First file exists: True
Output initialized.
Starting Parallel Processing (4 workers)...
Checkpoint 100/2147 | 161.4 files/min | ETA: 13 min
Checkpoint 200/2147 | 257.1 files/min | ETA: 8 mi

### **Merge Split-Level VAD Results**

After processing all spoof target splits individually, this section merges the generated VAD result files back into their corresponding manifests. The pipeline iterates over all processed splits, loads both the original manifest and the associated VAD result CSV, removes duplicated columns, and merges the VAD metadata using `file_id` as the matching key.

All merged split manifests are then concatenated into a single combined spoof target manifest:

```text
file_manifest_spoof_targets_ge3_with_vad.csv
```

This final combined manifest is later used in the main pipeline for spoof target filtering and validation.

In [ ]:
import os
import pandas as pd

# ============================================================
# Paths
# ============================================================

ge3_dir = (
    "/content/drive/MyDrive/age verification/"
    "processed/manifest/spoof_targets/ge3"
)

# ============================================================
# Splits
# ============================================================

splits = [
    'cv_test',
    'cv_val',
    'cv_train',
    'myst_train',
    'myst_test',
    'myst_val',
    'vox_train',
    'vox_test',
    'vox_val'
]

merged_dfs = []

# ============================================================
# Merge Manifest + VAD Results
# ============================================================

for split in splits:

    manifest_path = os.path.join(
        ge3_dir,
        f"file_manifest_{split}_ge3.csv"
    )

    vad_path = os.path.join(
        ge3_dir,
        f"file_manifest_{split}_ge3_vad_results.csv"
    )

    # --------------------------------------------------------
    # Skip if VAD results are missing
    # --------------------------------------------------------

    if not os.path.exists(vad_path):

        print(f"Skipping {split} (no VAD results)")
        continue

    # --------------------------------------------------------
    # Load CSVs
    # --------------------------------------------------------

    manifest_df = pd.read_csv(manifest_path)

    vad_df = pd.read_csv(vad_path)

    # --------------------------------------------------------
    # Remove duplicated columns from VAD results
    # --------------------------------------------------------

    vad_df = vad_df.drop(
        columns=[
            "source_path",
            "raw_duration_sec"
        ],
        errors="ignore"
    )

    # --------------------------------------------------------
    # Merge
    # --------------------------------------------------------

    merged_df = manifest_df.merge(
        vad_df,
        on="file_id",
        how="left"
    )

    print(
        f"{split} | "
        f"manifest={len(manifest_df)} | "
        f"vad={len(vad_df)} | "
        f"merged={len(merged_df)}"
    )

    merged_dfs.append(merged_df)

# ============================================================
# Combine All Splits
# ============================================================

file_manifest = pd.concat(
    merged_dfs,
    ignore_index=True
)

# ============================================================
# Final Info
# ============================================================

print("\nFinal shape:")
print(file_manifest.shape)

print("\nColumns:")
print(file_manifest.columns.tolist())

display(file_manifest.head())

cv_test | manifest=8148 | vad=8148 | merged=8148
cv_val | manifest=15360 | vad=15360 | merged=15360
cv_train | manifest=75951 | vad=75951 | merged=75951
myst_train | manifest=41105 | vad=41105 | merged=41105
myst_test | manifest=8048 | vad=8048 | merged=8048
myst_val | manifest=9272 | vad=9272 | merged=9272
vox_train | manifest=12182 | vad=12182 | merged=12182
vox_test | manifest=2762 | vad=2762 | merged=2762
vox_val | manifest=2147 | vad=2147 | merged=2147

Final shape:
(174975, 15)

Columns:
['file_id', 'speaker_id', 'dataset', 'split', 'pool', 'source_path', 'processed_path', 'raw_duration_sec', 'voiced_segs_sec', 'voiced_duration_sec', 'is_corrupted', 'speech_duration_sec', 'flagged', 'vad_status', 'max_amplitude']


,file_id,speaker_id,dataset,split,pool,source_path,processed_path,raw_duration_sec,voiced_segs_sec,voiced_duration_sec,is_corrupted,speech_duration_sec,flagged,vad_status,max_amplitude
0,common_voice_551d5ea36eb46269c7a7710255ff0cab3...,551d5ea36eb46269c7a7710255ff0cab38e68cd8360ad3...,common_voice,test,adult_spoof_targets,/content/drive/MyDrive/age verification/data/c...,/content/drive/MyDrive/age verification/proces...,5.520,[],0.0,False,3.55,True,success,0.771515
1,common_voice_551d5ea36eb46269c7a7710255ff0cab3...,551d5ea36eb46269c7a7710255ff0cab38e68cd8360ad3...,common_voice,test,adult_spoof_targets,/content/drive/MyDrive/age verification/data/c...,/content/drive/MyDrive/age verification/proces...,3.792,[],0.0,False,2.33,True,success,0.770081
2,common_voice_551d5ea36eb46269c7a7710255ff0cab3...,551d5ea36eb46269c7a7710255ff0cab38e68cd8360ad3...,common_voice,test,adult_spoof_targets,/content/drive/MyDrive/age verification/data/c...,/content/drive/MyDrive/age verification/proces...,4.464,[],0.0,False,2.81,True,success,0.617188
3,common_voice_551d5ea36eb46269c7a7710255ff0cab3...,551d5ea36eb46269c7a7710255ff0cab38e68cd8360ad3...,common_voice,test,adult_spoof_targets,/content/drive/MyDrive/age verification/data/c...,/content/drive/MyDrive/age verification/proces...,7.296,[],0.0,False,5.43,True,success,0.810913
4,common_voice_551d5ea36eb46269c7a7710255ff0cab3...,551d5ea36eb46269c7a7710255ff0cab38e68cd8360ad3...,common_voice,test,adult_spoof_targets,/content/drive/MyDrive/age verification/data/c...,/content/drive/MyDrive/age verification/proces...,4.368,[],0.0,False,2.91,True,success,0.595245


In [ ]:
# ============================================================
# Save Final Combined Manifest
# ============================================================

output_path = (
    "/content/drive/MyDrive/age verification/"
    "processed/manifest/spoof_targets/ge3/"
    "file_manifest_spoof_targets_ge3_with_vad.csv"
)

file_manifest.to_csv(
    output_path,
    index=False
)

print("\nSaved combined manifest:")
print(output_path)


Saved combined manifest:
/content/drive/MyDrive/age verification/processed/manifest/spoof_targets/ge3/file_manifest_spoof_targets_ge3_with_vad.csv
